# Example 4 -- Code Review & Safety Validation

Before executing any agent-generated code that touches a protected dataset,
the code reviewer scans for **prohibited data-access patterns** (row
iteration, direct file reads, individual value access, etc.).

**Use case:** an agent proposes analysis code to run on a sensitive dataset.
The privacy layer validates the code *before* execution, blocking anything
that would leak individual records.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from agent_privacy_layer import CodeReviewer, PrivacyLayer, UserConfirmation

reviewer = CodeReviewer()

## 1. Safe Code -- Aggregations and Schema Access

In [2]:
safe_code = """\
# Aggregation queries -- OK
mean_salary = df.groupby('department')['salary'].mean()
total = df['revenue'].sum()
desc = df.describe()
counts = df['status'].value_counts()

# Schema inspection -- OK
cols = df.columns
types = df.dtypes
num_rows = len(df)
"""

result = reviewer.review(safe_code)
print("=== Safe code ===")
print(result)

=== Safe code ===
Code review: APPROVED
  Violations : 0
  Warnings   : 0


## 2. Unsafe Code -- Row-Level Iteration

In [3]:
iteration_code = """\
for idx, row in df.iterrows():
    print(row['name'], row['salary'])
"""

result = reviewer.review(iteration_code)
print("=== Row iteration (blocked) ===")
print(result)

=== Row iteration (blocked) ===
Code review: REJECTED
  Violations : 1
  Warnings   : 0
  [ERROR] Line 1, Col 16: iterrows() exposes raw row data. Use aggregations or the DP layer.


## 3. Unsafe Code -- Direct File Reading

In [4]:
file_read_code = """\
import pandas as pd
data = pd.read_csv('sensitive_data.csv')
with open('secrets.txt') as f:
    contents = f.read()
"""

result = reviewer.review(file_read_code)
print("=== Direct file reads (blocked) ===")
print(result)

=== Direct file reads (blocked) ===
Code review: REJECTED
  Violations : 2
  Warnings   : 0
  [ERROR] Line 2, Col 7: Direct pd.read_csv() is prohibited. Use DataInspector or DPStatistics.
  [ERROR] Line 3, Col 5: Direct file open() is prohibited. Use the privacy layer to access data.


## 4. Unsafe Code -- Individual Value Access

In [5]:
value_access_code = """\
first_row = df.iloc[0]
specific_cell = df.loc[3, 'ssn']
raw_array = df.values
as_list = df['name'].to_list()
"""

result = reviewer.review(value_access_code)
print("=== Individual value access (blocked) ===")
print(result)

=== Individual value access (blocked) ===
Code review: REJECTED
  Violations : 5
  Warnings   : 0
  [ERROR] Line 1, Col 12: .iloc accesses individual rows/values. Use aggregations or the DP layer instead.
  [ERROR] Line 2, Col 16: .loc accesses individual rows/values. Use aggregations or the DP layer instead.
  [ERROR] Line 3, Col 12: .values exposes the underlying NumPy array of raw data.
  [ERROR] Line 4, Col 10: .to_list() exposes raw data.
  [ERROR] Line 4, Col 10: .to_list() exposes raw data.


## 5. Warnings -- head()/tail()

In [6]:
warning_code = """\
preview = df.head(5)
last_rows = df.tail(3)
"""

result = reviewer.review(warning_code)
print("=== head/tail warnings ===")
print(result)

=== head/tail warnings ===
Code review: APPROVED
  Violations : 0
  Warnings   : 2
  [WARNING] Line 1, Col 10: head() may expose raw data rows. Ensure only schema inspection is intended.
  [WARNING] Line 2, Col 12: tail() may expose raw data rows. Ensure only schema inspection is intended.


## 6. Using review_code() via PrivacyLayer

In [7]:
import pandas as pd

layer = PrivacyLayer(
    pd.DataFrame({"x": [1, 2, 3]}),
    confirmation=UserConfirmation(dry_run=True),
)

agent_code = "result = df.groupby('dept')['sales'].sum()"
review = layer.review_code(agent_code)
print("=== Via PrivacyLayer.review_code() ===")
print(f"Approved: {review.approved}")
print(f"Violations: {len(review.violations)}")
print(f"Warnings: {len(review.warnings)}")

=== Via PrivacyLayer.review_code() ===
Approved: True
Violations: 0
Warnings: 0


## Analysis

**Goal:** Validate agent-generated code before execution to block patterns that would leak individual records from a protected dataset.

**Results:**
- **Safe code** (aggregations like `groupby().mean()`, `sum()`, `describe()`, `value_counts()` and schema access like `df.columns`, `df.dtypes`) is correctly **APPROVED** with zero violations and zero warnings.
- **Row iteration** (`df.iterrows()`) is correctly **REJECTED** with 1 violation -- it exposes raw row data.
- **Direct file reads** (`pd.read_csv()`, `open()`) are correctly **REJECTED** with 2 violations -- they bypass the privacy layer entirely.
- **Individual value access** (`df.iloc[0]`, `df.loc[3, 'ssn']`, `df.values`, `df['name'].to_list()`) is correctly **REJECTED** with 5 violations -- all patterns expose raw data.
- **head()/tail()** are **APPROVED** with 2 warnings -- a sensible middle ground since these may or may not be safe depending on context.
- **PrivacyLayer integration** via `review_code()` works correctly, providing the same review functionality through the main API.

**Verdict:** The code reviewer achieves its goal. It correctly distinguishes between safe aggregate operations and unsafe individual-data-access patterns. The warning system for ambiguous cases (head/tail) is a thoughtful design choice that avoids being overly restrictive while still flagging potential risks.